In [2]:
pip install gurobipy

Note: you may need to restart the kernel to use updated packages.


In [10]:
#Lê instâncias rotas não lineares 
#Arquivos disponíveis em: https://www.ime.usp.br/~egbirgin/sources/briro2024/
#Descrição das instâncias: E. G. Birgin, P. Feofiloff, C. G. Fernandes, E. L. de Melo, M. T. I. Oshiro, and D. P. Ronconi.A MILP model for an extended version of the flexible job shop problem. Optimization Letters, 8(4):1417–1431, 2014.
import math
def ler_instancia_txt(caminho_arquivo, g):
    with open(caminho_arquivo, 'r') as f:
        linhas = f.readlines()
    #print(f'Lendo arquivo')

    # n #máquinas, o #operações
    o, num_precedencias, n = map(int, linhas[0].split())
    oper_linhas = linhas[1 + num_precedencias : 1 + num_precedencias + o]
    FFp = linhas[1 + num_precedencias + o]
    #print(f'FFp {FFp}, tipo {type(FFp)} e repr {repr(FFp)}')
    #print(f'primeira linha {o}, {num_precedencias} e {n}')
    #print(f'linhas operações por máquina {oper_linhas}')

# A
    #A: pares de operações que são imediatamente precedentes
    A = []
    for i in range(1, 1 + num_precedencias):
        i1, i2 = map(int, linhas[i].split())
        A.append([i1+1, i2+1])  # indices originais

    
    todas_operacoes = set()
    sucessores = set()
    for i, j in A:
        todas_operacoes.add(i)
        todas_operacoes.add(j)
        sucessores.add(j)
    operacoes_iniciais = todas_operacoes - sucessores
    
    for v in operacoes_iniciais:
        A.append([0, v])
    A.sort(key=lambda x: (x[0], x[1]))
    #print(f'Sequencias {A}') 
    
# Fv, Vk e pik
    #Fv: máquinas que processam operação v 
    #Vk: operações que podem ser processados pela máquina k
    #pik: tempo de processamento, se a máquina k não pode processar a operação i, então pik=0 
    
    #inicializando
    Fv = [[] for _ in range(o+1)]  # INCLUI OPERAÇÃO DUMMY
    Vk = [[] for _ in range(n)] 
    pik = [[0 for _ in range(n)] for _ in range(o+1)] #INCLUI OPERAÇÃO DUMMY
    
    #ajuste para operação 0
    Fv[0] = list(range(0, n))
    for k in range(n):
        Vk[k].append(0)
        pik[0][k] = 0  # tempo zero para dummy
        
    #preenchendo com dados do txt
    for idx, linha in enumerate(oper_linhas, start=1):
        partes = list(map(int, linha.strip().split()))
        num_mqs = partes[0]
        for i in range(num_mqs):
            k = partes[1 + 2 * i]
            p = partes[2 + 2 * i]
            Fv[idx].append(k)
            Vk[k].append(idx)
            pik[idx][k] = p
    #print(f'Fv - máquinas que processam operação v: {Fv} \n Vk - operações que podem ser processados pela máquina k: {Vk} \n pik - tempo de processamento: {pik} ')

    #B e Bk
    #Bk: pares possíveis de operações que podem ser processadas na máquina k
    #B: Todos os pares do conjunto Bk
    
    Bk = [[] for _ in range(n)]  # índice 1 a n
    for k in range(n):
        pares = [[v, w] for v in Vk[k] for w in Vk[k] if v != w]
        Bk[k] = pares
    
    B_tudo = []
    for pares in Bk[0:]:
        B_tudo.extend(pares)
        B = sorted([list(t) for t in dict.fromkeys(tuple(x) for x in B_tudo)], key=lambda par: (par[0], par[1]))

    #print(f'Bk {Bk} \n B {B}')
#C
    #C: pares de operações que precedem uma mesma operação
    predecessores_por_c = {}
    for a, c in A:
        if c not in predecessores_por_c:
            predecessores_por_c[c] = []
        predecessores_por_c[c].append(a)
    
    C = []
    for c, preds in predecessores_por_c.items():
        preds = sorted(set(preds))
        for i in range(len(preds)):
            for j in range(i + 1, len(preds)):
                C.append([preds[i], preds[j]])
    #print(f'C {C}')
    
# Vf
    #Vf:operações finais
    operacoes_com_sucessor = {i for (i, _) in A}
    #print(f'operacoes_com_sucessor: {operacoes_com_sucessor}')
    Vf = [v for v in range(o+1) if v not in operacoes_com_sucessor]
    #print(f'Vf {Vf}')

#dd
    #dd: data de entrega para cada job    
    valores = FFp.strip().split()
    valores_numericos = [v for v in valores if v != '.' and v != '']
    datas = list(map(float, valores_numericos))
    #print(f'datas {datas}')

    datas = [math.floor(x * g) for x in datas]
    
    dd=[0]* len(range(o+1))
    for v, t in zip(Vf, datas):
        dd[v] = t

    return {
        'n': n,
        'o': o,
        'A': A,
        'Fv': Fv,
        'Vk': Vk,
        'Bk': Bk,
        'B': B,
        'pik': pik,
        'C': C,
        'Vf': Vf,
        'dd': dd
    }



In [11]:
#Modelo MILP - BFJS com minimização de atraso e rotas não lineares

def resolver_instancia(dados, nome_saida):
    
    V = range(dados['o']+1) #início em 0 com o elementos (inclui operação dummy)
    M = range(dados['n']) #início em 0 com n elementos
    A = dados['A']
    B = dados['B']
    Bk = dados['Bk']
    Fv = dados['Fv'] #ajusta os valores de Fv para começar em 0. 
    Vf = dados['Vf']
    pik = dados['pik']
    C = dados['C']
    dd = dados['dd']

    L = sum(max(pik[i][k] for k in Fv[i]) for i in V)
    
    mod=gp.Model()
        
    #VARIÁVEIS
    X=mod.addVars(V,M,vtype=gp.GRB.BINARY, name="X" ) #1, se a operação v  ́e processada na máquina k; 0, caso contrário
    Y=mod.addVars(V,V,vtype=gp.GRB.BINARY, name="Y") #1, se a operação v vem antes da operação w onde (v, w) ∈ B; 0, caso contrário
    P=mod.addVars(V, name="P") #tempo de processamento da operação v se processada na máquina escolhida (p')
    D=mod.addVars(V, name="D") #instante de partida da operação v (departure time) *********NOVO
    T=mod.addVars(Vf, name="T") #atraso da operação v e Vf
        
    #FUNÇÃO OBJETIVO
    mod.setObjective(gp.quicksum(T[v] for v in Vf), sense=gp.GRB.MINIMIZE) #FO de minimização do atraso
            
    #RESTRIÇÕES
    r1=mod.addConstrs(gp.quicksum(X[v,k] for k in Fv[v])==1 for v in V)
    r2=mod.addConstrs(P[v]==gp.quicksum(X[v,k]*pik[v][k] for k in Fv[v]) for v in V)
    r3=mod.addConstrs(D[v]+P[w]-(1-Y[v,w])*L<=D[w] for (v,w) in B)
    r4=mod.addConstrs(Y[v,w]+Y[w,v]>=X[v,k]+X[w,k]-1 for k in M for (v,w) in Bk[k])
    r5=mod.addConstrs(D[v]+P[w]<=D[w] for (v,w) in A)
    r6=mod.addConstrs(T[v]>=D[v]-dd[v] for v in Vf)
    r7=mod.addConstrs(D[v]>=D[w]-(3-Y[w,u]-X[w,k]-X[u,k])*L for k in M for (w,u) in Bk[k] 
                            for (v,u2) in A if v!=0 and w!=u and w!=v and u==u2)
    r8=mod.addConstr(D[0]==0) #*********NOVO
    r9=mod.addConstrs(D[i]==D[j] for (i,j) in C) #garante que se (i,h) e (j,h) \in A então Di=Dj - i e j antecedem a mesma operação
            
    caminho_lp = os.path.join(DIR_SAIDA, f"{nome_saida}.lp")
    #print(f"DIR_SAIDA: {DIR_SAIDA}")
    #print(f"nome_saida: {nome_saida}")
    #print(f"caminho_lp: {caminho_lp}")
    #print(f"Tipo de caminho_lp: {type(caminho_lp)}")
    
    #mod.setParam('TimeLimit', 3600) #O tempo limite não deve ser definido no .lp, mas no arquivo do águia!

    mod.write(caminho_lp)    
    
    #mod.optimize()
    #print("Função objetivo:", mod.ObjVal)
    #print("Número de variáveis:", mod.NumVars)
    #print("Número de variáveis inteiras ou binárias:", sum(1 for v in mod.getVars() if v.VType in ['I', 'B']))
    #print("Número de restrições:", mod.NumConstrs)
    #print("Tempo de processamento (segundos):", mod.Runtime)

    return {
        'modelo': mod,
        'X': X,
        'Y': Y,
        'D': D,
        'P': P,
        'T': T,
        'Vf': dados['Vf'],
        'V': range(dados['o']+1),  
        'M': range(dados['n']),
        'A': dados['A'],
        'pik': dados['pik'],
        'dd': dados['dd']
    }


In [15]:
import gurobipy as gp
import glob
import os

from pathlib import Path


DIR_INSTANCIAS = "C:/Users/Hanna Pamplona/Doutorado/Instâncias/non-linear routes/com FIFO/Y_FFp"
DIR_SAIDA = "C:/Users/Hanna Pamplona/Doutorado/"

valores_g = [0.25, 0.5, 0.75]
tipos = ["tight", "moderate", "loose"]


for g, tipo in zip(valores_g, tipos):
    #print(f"\n=== Processando g={g}, tipo={tipo} ===")
    
    for arquivo in glob.glob(os.path.join(DIR_INSTANCIAS, "*.txt")):
        nome_base = os.path.basename(arquivo).replace("_FFp.txt", "")
        nome_saida = f"MILP_Birgin_{nome_base}_{tipo}"  
        print(f"Processando: {nome_saida}")    
        
        try:
            dados = ler_instancia_txt(arquivo, g)
            resultados = resolver_instancia(dados, nome_saida)
            #print(f'AQUI')
        except Exception as e:
            print(f"Erro em {nome_saida}: {str(e)}")


Processando: MILP_Birgin_YFJS01_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS02_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS03_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS04_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS05_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS06_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS07_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS08_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS09_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS10_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS11_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJS12_tight
Set parameter TimeLimit to value 3600
AQUI
Processando: MILP_Birgin_YFJ

In [22]:
import os
import glob

def salvar_resultados_txt(resultados, caminho):
    with open(caminho, 'w') as f:
        m = resultados['modelo']
        X = resultados['X']
        Y = resultados['Y']
        D = resultados['D']
        T = resultados['T']
        V = resultados['V']
        M = resultados['M']
        Vf = resultados['Vf']
        dd = resultados['dd']

        f.write(f"Status: {m.status}\n")
        f.write(f"Objetivo: {m.ObjVal}\n")
        f.write(f"Bound: {m.ObjBound}\n")
        f.write(f"Tempo: {m.Runtime:.2f}s\n")
        f.write(f"GAP: {m.MIPGap*100:.2f}%\n\n")
        
        # Variáveis X (ajustado para usar dados['X'])
        for v in V:
            for k in M:
                if X[v,k].x > 0.5:
                    f.write(f"X{v},{k}: {X[v,k].x}\n")
        
        # Variáveis Y
        f.write("\n")
        for v in V:
            for w in V:
                if Y[v,w].x > 0.5 and v != w:
                    f.write(f"Y{v},{w}: {Y[v,w].x}\n")
        
        # Tempos D e T
        f.write("\n")
        for v in V:
            f.write(f"D[{v}]={D[v].x}\n")
        
        f.write("\n")
        for v in Vf:
            f.write(f"T[{v}]={T[v].x}\n")
            f.write(f'dd[{v}]={dd[v]}\n')


def salvar_gantt(resultados, caminho):
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    from collections import defaultdict, deque
    
    # Extrai todas variáveis necessárias do dicionário
    X = resultados['X']
    D = resultados['D']
    P = resultados['P']
    A = resultados['A']
    Vf = resultados['Vf']
    V = resultados['V']
    M = resultados['M']
    
    def identificar_jobs(A):
        grafo = defaultdict(list)
        for v, w in A:
            grafo[v].append(w)
        
        job_por_operacao = {}
        job_id = 1
        
        for raiz in grafo[0]:
            fila = deque([raiz])
            while fila:
                atual = fila.popleft()
                if atual in job_por_operacao:
                    continue
                job_por_operacao[atual] = job_id
                for vizinho in grafo.get(atual, []):
                    fila.append(vizinho)
            job_id += 1
        
        return job_por_operacao
    
    def job_de_operacao_automatico(op, Vf):
        for i, limite in enumerate(Vf):
            if op <= limite:
                return i + 1
        return 0
    
    # Configurações
    escala = 600
    cores_disponiveis = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red',
                        'tab:purple', 'tab:brown', 'tab:pink', 'tab:olive',
                        'tab:cyan', 'tab:gray']
    
    # Cores por job
    cores_job = {i+1: cores_disponiveis[i % len(cores_disponiveis)] 
                for i in range(len(Vf))}
    
    # Monta dados para Gantt
    dados_gantt = []
    for v in V:
        for k in M:
            if X[v,k].x > 0.5:  # Verifica se a operação foi alocada
                inicio = D[v].x - P[v].x
                fim = D[v].x
                dados_gantt.append((v, k, inicio, fim))
    
    # Cria figura
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Plotagem das operações
    for op, maq, ini, fim in dados_gantt:
        job = job_de_operacao_automatico(op, Vf)
        cor = cores_job.get(job, 'lightgray')
        ax.add_patch(
            patches.Rectangle(
                (ini/escala, maq-0.4),
                (fim-ini)/escala,
                0.8,
                edgecolor='black',
                facecolor=cor
            )
        )
        ax.text(
            (ini + fim)/(2*escala),
            maq,
            f"Op{op}",
            ha='center',
            va='center',
            color='white',
            fontsize=8
        )
    
    # Configurações do gráfico
    maquinas = sorted(set(m for _,m,_,_ in dados_gantt))
    ax.set_yticks(maquinas)
    ax.set_yticklabels([f"Máquina {k}" for k in maquinas])
    ax.set_xlabel("Tempo")
    ax.set_ylabel("Máquinas")
    ax.set_title("Gráfico de Gantt - Modelo MILP")
    ax.grid(True, axis='x')
    
    todas_maquinas = sorted(set(m for _,m,_,_ in dados_gantt))
    if todas_maquinas:  # Só ajusta se houver máquinas
        margem = 0.8  # Espaço extra acima/abaixo
        ax.set_ylim(
            min(todas_maquinas) - margem,
            max(todas_maquinas) + margem
        )
    
    # Configurações adicionais para evitar corte
    ax.set_yticks(todas_maquinas)
    ax.set_yticklabels([f"Máquina {m}" for m in todas_maquinas]) 
    
    plt.subplots_adjust(
        top=0.95,
        bottom=0.1,
        left=0.15,  # Aumentei a margem esquerda
        right=0.95
    )

    
    
    plt.tight_layout()
    plt.savefig(caminho, dpi=300, bbox_inches='tight')
    plt.close()